
# Phase 1 — Foundations of Time Series Analysis with Python  
## Beginner-friendly, practical, and engaging

**Course:** Data Science with Python  
**Module:** Time Series Analysis  
**Phase 1 Focus:** Foundations  
**Suggested duration:** 6 hours  

---

## Why this notebook matters

Time series analysis is different from ordinary machine learning because **time has structure**.

In a normal dataset, rows can often be shuffled.  
In a time series dataset, that is usually a mistake.

This notebook helps students build the right mental model first.

---

## Learning goals

By the end of this notebook, students should be able to:

- explain what a time series is
- explain how time series differs from ordinary tabular data
- work with dates and time indexes in pandas
- sort, index, and filter time-based data correctly
- understand time frequency such as daily, weekly, and monthly data
- detect and handle missing timestamps
- resample time series data
- visualize trend, seasonality, and noise
- understand additive vs multiplicative patterns
- use decomposition to separate components
- explain the idea of stationarity in intuitive terms
- apply differencing and moving averages
- prepare for later forecasting modules

---

## Recommended teaching structure for a 6-hour session

### Hour 1
- What is time series?
- Why time order matters
- Real-world examples

### Hour 2
- Working with datetime in pandas
- Sorting, indexing, slicing
- Understanding frequency

### Hour 3
- Plotting time series
- Trend, seasonality, and noise
- Moving averages

### Hour 4
- Resampling and aggregation
- Missing dates
- Daily vs weekly vs monthly views

### Hour 5
- Additive vs multiplicative behavior
- Time series decomposition

### Hour 6
- Stationarity intuition
- Differencing
- Practical review and exercises


In [ ]:

# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.seasonal import seasonal_decompose

%matplotlib inline



# Part 1: What is a time series?

A **time series** is a sequence of observations recorded over time.

Examples:
- daily sales
- monthly revenue
- hourly electricity usage
- weekly product demand
- stock prices
- website traffic

In this notebook, we will use a practical retail-style dataset with:

- `sales`
- `orders`
- `ad_spend`
- `website_visits`

**File used:** `time_series_foundations_retail_daily.csv`


In [ ]:

# Load the dataset
df = pd.read_csv("/mnt/data/time_series_foundations_retail_daily.csv")

# Show the first 5 rows
df.head()



## First observation

At first glance, this looks like a normal table.

But it is not just a table.

The `date` column changes everything, because now:

- order matters
- future values should not help predict the past
- nearby observations may be related
- patterns such as weekends and holidays may matter


In [ ]:

# Basic shape and data types
print("Rows and columns:", df.shape)
print()
print(df.dtypes)



# Part 2: Convert dates properly

In time series analysis, dates should not stay as plain text.

We usually convert them into pandas datetime format.


In [ ]:

# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Check data types again
df.dtypes



## Why `datetime` matters

Once dates are stored properly, we can do useful things like:

- sort by time
- extract month or weekday
- filter by date range
- resample by week or month
- set date as the index


In [ ]:

# Sort by date just to be safe
df = df.sort_values("date")

# Set date as the index
df = df.set_index("date")

# Show the first few rows
df.head()



## Why set the date as the index?

This makes time-based operations much easier.

For example:
- `df.loc["2024-01"]` for one month
- `df.resample("W").sum()` for weekly totals
- `df.asfreq("D")` for daily frequency


In [ ]:

# Quick examples of time-based slicing
print("January 2024")
display(df.loc["2024-01"].head())

print("A short date range")
display(df.loc["2024-03-01":"2024-03-07"])



# Part 3: First time series plots

Time series should be plotted against time first.

This is the fastest way to start seeing:
- long-term movement
- repeating patterns
- unusual spikes
- possible missing periods


In [ ]:

# Plot daily sales
plt.figure(figsize=(12, 4))
plt.plot(df.index, df["sales"])
plt.xlabel("Date")
plt.ylabel("Sales")
plt.title("Daily Sales Over Time")
plt.show()



## What should students notice?

Likely observations:
- the line is noisy from day to day
- the overall level seems to grow over time
- there may be repeating patterns
- some periods may look higher than others

These ideas lead to the core time series components.



# Part 4: Time series components

A time series often contains three important parts:

### 1. Trend
The long-term direction of the series  
Example: sales slowly increasing over time

### 2. Seasonality
A repeating pattern  
Example: higher sales on weekends or holidays

### 3. Noise
Random variation that is hard to explain exactly

A simple mental model is:

**series = trend + seasonality + noise**


In [ ]:

# Plot a second variable for comparison
plt.figure(figsize=(12, 4))
plt.plot(df.index, df["orders"])
plt.xlabel("Date")
plt.ylabel("Orders")
plt.title("Daily Orders Over Time")
plt.show()



## Teaching tip

Ask students:
- Do sales and orders move together?
- Do you see weekly patterns?
- Does the business appear to be growing?
- Are some months stronger than others?

This gets them thinking like analysts, not just coders.



# Part 5: Rolling averages

Daily data can be noisy.

A **rolling average** smooths short-term noise so we can see the bigger pattern more clearly.

For example:
- a 7-day rolling average often helps with daily retail data


In [ ]:

# Create a 7-day rolling average for sales
df["sales_roll7"] = df["sales"].rolling(7).mean()

plt.figure(figsize=(12, 4))
plt.plot(df.index, df["sales"], alpha=0.5, label="Daily Sales")
plt.plot(df.index, df["sales_roll7"], label="7-Day Rolling Average")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.title("Daily Sales with 7-Day Rolling Average")
plt.legend()
plt.show()



## Why rolling averages help

They help students see:
- the underlying trend more clearly
- whether the series is rising or falling
- whether short-term noise was hiding the pattern



# Part 6: Extract time-based features

Even in the foundations phase, students should learn that dates contain useful information.

Examples:
- year
- month
- day of week
- quarter
- whether a day is a weekend


In [ ]:

# Create simple time features
df["year"] = df.index.year
df["month"] = df.index.month
df["day_of_week"] = df.index.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

df[["sales", "year", "month", "day_of_week", "is_weekend"]].head()


In [ ]:

# Average sales by day of week
dow_avg = df.groupby("day_of_week")["sales"].mean()

plt.figure(figsize=(7, 4))
plt.bar(dow_avg.index.astype(str), dow_avg.values)
plt.xlabel("Day of Week (0=Mon, 6=Sun)")
plt.ylabel("Average Sales")
plt.title("Average Sales by Day of Week")
plt.show()



## Why this matters

This helps students see that time itself contains information.

A date is not just a label.  
It can help explain behavior.



# Part 7: Frequency

Time series can have different frequencies:

- hourly
- daily
- weekly
- monthly
- quarterly
- yearly

Frequency matters because it affects:
- what patterns we can see
- what model we should use
- how we define seasonality


In [ ]:

# Inspect whether our index looks daily
df.index[:10]



## Important practical issue

Sometimes a dataset *should* be daily, but some dates are missing.

That means:
- the series may not have a clean daily frequency
- plots may still look okay
- but resampling and decomposition may become harder

So we should check for missing timestamps.


In [ ]:

# Create a complete daily date range from min to max date
full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq="D")

print("Expected number of daily dates:", len(full_range))
print("Actual number of rows:", len(df))
print("Number of missing dates:", len(full_range.difference(df.index)))


In [ ]:

# Show the missing dates
full_range.difference(df.index)



# Part 8: Handling missing timestamps

One common step is to force the series to a regular frequency using `asfreq()`.

This inserts missing dates and shows them explicitly as `NaN`.


In [ ]:

# Force daily frequency
df_daily = df.asfreq("D")

# Check the first rows around a missing period
df_daily.head(15)


In [ ]:

# Count missing values after forcing daily frequency
df_daily.isna().sum()



## Why this is useful

Before `asfreq("D")`, missing dates were invisible.  
After `asfreq("D")`, we can see exactly where data is missing.

That makes later cleaning steps much more honest.



## Simple ways to handle missing time points

Options include:
- leave them missing for now
- fill them with interpolation
- forward fill / backward fill
- fill with domain-specific rules

For foundations, we will use interpolation for a simple demo.


In [ ]:

# Interpolate missing numeric values
df_filled = df_daily.interpolate(method="time")

# Check missing values again
df_filled.isna().sum()


In [ ]:

# Plot filled sales series
plt.figure(figsize=(12, 4))
plt.plot(df_filled.index, df_filled["sales"])
plt.xlabel("Date")
plt.ylabel("Sales")
plt.title("Daily Sales After Filling Missing Dates")
plt.show()



# Part 9: Resampling

Resampling changes the frequency of a time series.

Examples:
- daily → weekly
- daily → monthly

This is extremely common in business analysis.


In [ ]:

# Weekly total sales
weekly_sales = df_filled["sales"].resample("W").sum()

plt.figure(figsize=(12, 4))
plt.plot(weekly_sales.index, weekly_sales.values)
plt.xlabel("Week")
plt.ylabel("Weekly Sales")
plt.title("Weekly Total Sales")
plt.show()


In [ ]:

# Monthly average sales
monthly_avg_sales = df_filled["sales"].resample("M").mean()

plt.figure(figsize=(10, 4))
plt.plot(monthly_avg_sales.index, monthly_avg_sales.values, marker="o")
plt.xlabel("Month")
plt.ylabel("Average Monthly Sales")
plt.title("Average Monthly Sales")
plt.show()



## Teaching point

When we resample:
- weekly totals answer one kind of business question
- monthly averages answer another kind of business question

Students should learn that aggregation changes the meaning.



# Part 10: Additive vs multiplicative patterns

Two common ways to think about time series structure are:

### Additive
\[
y = trend + seasonality + noise
\]

### Multiplicative
\[
y = trend \times seasonality \times noise
\]

---

## Intuition

### Additive pattern
Seasonal ups and downs stay about the same size over time

### Multiplicative pattern
Seasonal ups and downs grow as the series level grows


In [ ]:

# Plot monthly average sales again to think about structure
plt.figure(figsize=(10, 4))
plt.plot(monthly_avg_sales.index, monthly_avg_sales.values, marker="o")
plt.xlabel("Month")
plt.ylabel("Average Monthly Sales")
plt.title("Thinking About Additive vs Multiplicative Behavior")
plt.show()



## Beginner rule of thumb

- If seasonal swings look roughly constant → additive may be reasonable
- If seasonal swings get larger as the level grows → multiplicative may be reasonable



# Part 11: Time series decomposition

Decomposition helps us separate a series into:

- trend
- seasonality
- residual / noise

This is one of the most visual and intuitive tools in time series analysis.


In [ ]:

# Decompose the sales series using a weekly seasonality guess (period=7)
decomp = seasonal_decompose(df_filled["sales"], model="additive", period=7)

decomp.plot()
plt.show()



## How to read the decomposition plot

### Observed
The original time series

### Trend
The smoothed long-term movement

### Seasonal
The repeating pattern

### Resid
What is left after removing trend and seasonality


In [ ]:

# Try a multiplicative decomposition too
decomp_mult = seasonal_decompose(df_filled["sales"], model="multiplicative", period=7)

decomp_mult.plot()
plt.show()



## Discussion prompt

Ask students:
- Which decomposition seems more reasonable here?
- Does the seasonal pattern seem stable?
- Does the trend look smooth?



# Part 12: Stationarity intuition

Stationarity is a very important idea in time series.

A time series is **stationary** if its statistical behavior is relatively stable over time.

In simple beginner terms:
- the average level does not keep drifting
- the variance does not keep changing a lot
- the pattern is more stable

Many classical time series models work better when the series is closer to stationary.



## Non-stationary examples

A series may be non-stationary because of:
- trend
- seasonality
- changing variance
- structural changes

Our sales series likely has at least:
- trend
- seasonality


In [ ]:

# Compare original series and first difference
df_filled["sales_diff_1"] = df_filled["sales"].diff()

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(df_filled.index, df_filled["sales"])
axes[0].set_title("Original Sales Series")
axes[0].set_ylabel("Sales")

axes[1].plot(df_filled.index, df_filled["sales_diff_1"])
axes[1].set_title("First Difference of Sales")
axes[1].set_ylabel("Differenced Sales")

plt.xlabel("Date")
plt.show()



## What is differencing?

Differencing means subtracting the previous value:

\[
y_t - y_{t-1}
\]

This often helps remove trend and make a series more stable.


In [ ]:

# A 7-day seasonal difference
df_filled["sales_diff_7"] = df_filled["sales"] - df_filled["sales"].shift(7)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(df_filled.index, df_filled["sales"])
axes[0].set_title("Original Sales Series")
axes[0].set_ylabel("Sales")

axes[1].plot(df_filled.index, df_filled["sales_diff_7"])
axes[1].set_title("7-Day Seasonal Difference")
axes[1].set_ylabel("Sales Difference")

plt.xlabel("Date")
plt.show()



## Why seasonal differencing can help

If a pattern repeats every 7 days, then:

\[
y_t - y_{t-7}
\]

can help remove weekly seasonality.



# Part 13: Why time series train/test split must be chronological

In ordinary machine learning, random train/test split is common.

In time series, that is often wrong because it mixes past and future.

Correct idea:
- train on earlier dates
- test on later dates


In [ ]:

# Chronological split example
train = df_filled.loc[: "2024-06-30"]
test = df_filled.loc["2024-07-01":]

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Train last date:", train.index.max())
print("Test first date:", test.index.min())



## This matters because

A forecasting model should only learn from the past.

If future information leaks into training, model performance will look better than it really is.



# Part 14: Mini review of key concepts

At this point, students should know:

- how to convert and index dates
- how to visualize a time series
- what trend, seasonality, and noise mean
- how rolling averages help
- how to check for missing timestamps
- how to resample data
- what decomposition does
- what stationarity means in simple terms
- why differencing helps
- why train/test split must respect time



# Part 15: Exercises

## Exercise 1
Plot `website_visits` over time.  
What patterns do you notice?

## Exercise 2
Create a 14-day rolling average for sales.

## Exercise 3
Resample sales to monthly totals instead of monthly averages.

## Exercise 4
Group by month and compute average sales by month number.

## Exercise 5
Create a column for quarter.

## Exercise 6
Compute the first difference for `orders`.

## Exercise 7
Explain in plain English:
- trend
- seasonality
- noise
- stationarity
- differencing


In [ ]:

# Starter code for Exercise 2
df_filled["sales_roll14"] = df_filled["sales"].rolling(14).mean()

plt.figure(figsize=(12, 4))
plt.plot(df_filled.index, df_filled["sales"], alpha=0.5, label="Daily Sales")
plt.plot(df_filled.index, df_filled["sales_roll14"], label="14-Day Rolling Average")
plt.legend()
plt.title("Exercise: 14-Day Rolling Average")
plt.show()


In [ ]:

# Starter code for Exercise 3
monthly_total_sales = df_filled["sales"].resample("M").sum()
monthly_total_sales.head()



# Part 16: Key takeaways

- Time series data must be treated differently because **time order matters**
- Dates should be converted to `datetime` and often used as the index
- Good first steps are plotting, slicing, and checking frequency
- Trend, seasonality, and noise are the core building blocks
- Rolling averages help reveal the bigger picture
- Missing timestamps should be checked explicitly
- Resampling is essential for moving between daily, weekly, and monthly views
- Decomposition helps students visualize the structure of a series
- Stationarity means the series behaves more consistently over time
- Differencing can help make a series more stable
- Time-based train/test split is essential to avoid leakage



## End of Phase 1

This notebook focused on foundations.

A natural Phase 2 would cover:
- lag features
- forecasting baselines
- time series feature engineering
- machine learning for forecasting
- forecast evaluation metrics
